In [167]:
import re
from langchain_community.document_loaders import PyMuPDFLoader
import os

In [168]:
FOLDER_PATH = '../data/raw_pdf'
FILE_PATH = os.path.join(FOLDER_PATH, 'DGI_2024.pdf')

In [209]:
import yaml

with open('../src/config.yml', 'r') as f:
    config = yaml.safe_load(f)

In [211]:
config['FOLDER_PATH']

'../data/raw_pdf'

In [169]:
def clean_text(text):
    # Remove non-breaking spaces (\xa0)
    text = text.replace('\xa0', ' ')

    # Remove multiple spaces/tabs but preserve newlines
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove excessive newlines
    text = re.sub(r'\n+', '\n', text)

    # Remove page numbers
    text = re.sub(r'Page \d+', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    return text.strip()

In [170]:

for filename in os.listdir(FOLDER_PATH):
    if filename.endswith('.pdf'):
        file_path = os.path.join(FOLDER_PATH, filename)
        loader = PyMuPDFLoader(file_path=file_path)
        print(f"Loading {filename}...")
        docs = loader.load()
        cleaned_docs = [clean_text(doc.page_content) for doc in docs] 
        print(f"Loaded {len(cleaned_docs)} documents from {filename}")

Loading DGI_2024.pdf...
Loaded 148 documents from DGI_2024.pdf
Loading DietaryGuidelinesforNINwebsite.pdf...
Loaded 139 documents from DietaryGuidelinesforNINwebsite.pdf
Loading healthy-diet-fact-sheet-394.pdf...
Loaded 6 documents from healthy-diet-fact-sheet-394.pdf


In [171]:
cleaned_docs[0]

'1\nFACT SHEET N°394 \nUPDATED AUGUST 2018\nHealthy diet\nKEY FACTS\nn A healthy diet helps to protect against malnutrition in all its forms, as well as noncommunicable diseases \n(NCDs) such as diabetes, heart disease, stroke and cancer.\nn Unhealthy diet and lack of physical activity are leading global risks to health.\nn Healthy dietary practices start early in life – breastfeeding fosters healthy growth and improves cognitive \ndevelopment, and may have longer term health benefits such as reducing the risk of becoming overweight \nor obese and developing NCDs later in life.\nn Energy intake (calories) should be in balance with energy expenditure. To avoid unhealthy weight gain, total \nfat should not exceed 30% of total energy intake (1, 2, 3). Intake of saturated fats should be less than 10% of \ntotal energy intake, and intake of trans-fats less than 1% of total energy intake, with a shift in fat consumption \naway from saturated fats and trans-fats to unsaturated fats (3), and t

In [172]:
### Fixed size chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
combined_text = "\n\n".join(cleaned_docs)
chunks = splitter.split_text(combined_text)

In [173]:
len(chunks)

23

In [174]:
## Semantic chunking
import ollama
import numpy as np
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity

In [175]:
sentences = sent_tokenize(combined_text)

In [176]:
def convert_embeddings(prompt):
    response = ollama.embeddings(model="nomic-embed-text:v1.5", prompt=prompt)
    return np.array(response['embedding'])

In [177]:
# Generate embeddings using Ollama
char_embeddings = []

for sentence in chunks:
    embedding = convert_embeddings(prompt=sentence)
    char_embeddings.append(embedding)

In [178]:
# Generate embeddings using Ollama
sent_embeddings = []

for sentence in sentences:

    response = ollama.embeddings(
        model='nomic-embed-text:v1.5',
        prompt=sentence
    )

    sent_embeddings.append(response["embedding"])

In [179]:
# Semantic chunking
THRESHOLD = 0.75

similarity_chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):

    similarity = cosine_similarity(
        [sent_embeddings[i - 1]],
        [sent_embeddings[i]]
    )[0][0]
    
    # Topic changed
    if similarity < THRESHOLD:

        similarity_chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

    else:
        current_chunk.append(sentences[i])

# Add final chunk
similarity_chunks.append(" ".join(current_chunk))

In [180]:
len(similarity_chunks)

138

In [181]:
len(chunks)

23

In [183]:
semantic_embeddings = []
for s_chunks in similarity_chunks:
    embeddings = convert_embeddings(prompt=s_chunks)
    semantic_embeddings.append(embeddings)

In [184]:
print("#"*30, "Recursive Character Chunking", "#"*30)
print(f"Number of chunks: {len(chunks)}")
print(f"Average chunk length: {np.mean([len(chunk) for chunk in chunks]):.2f} characters")
print(f"embeddings shape: {np.array(char_embeddings).shape}")
print(f"embedding total count: {len(char_embeddings)}")

print("\n\n", "#"*30, "Semantic Chunking", "#"*30)
print(f"Number of chunks: {len(similarity_chunks)}")
print(f"Average chunk length: {np.mean([len(chunk) for chunk in similarity_chunks]):.2f} characters")
print(f"embeddings shape: {np.array(semantic_embeddings).shape}")
print(f"embedding total count: {len(semantic_embeddings)}")

############################## Recursive Character Chunking ##############################
Number of chunks: 23
Average chunk length: 833.61 characters
embeddings shape: (23, 768)
embedding total count: 23


 ############################## Semantic Chunking ##############################
Number of chunks: 138
Average chunk length: 134.46 characters
embeddings shape: (138, 768)
embedding total count: 138


'To avoid unhealthy weight gain, total \nfat should not exceed 30% of total energy intake (1, 2, 3). Intake of saturated fats should be less than 10% of \ntotal energy intake, and intake of trans-fats less than 1% of total energy intake, with a shift in fat consumption \naway from saturated fats and trans-fats to unsaturated fats (3), and towards the goal of eliminating industrially-\nproduced trans-fats (4, 5, 6).'

In [185]:
from pymilvus import connections

connections.connect(
    alias="default",
    host="localhost",
    port="19530"
)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\2366468844.py:3: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(


In [186]:
from pymilvus import (
    FieldSchema,
    CollectionSchema,
    DataType,
    Collection
)

fields = [

    FieldSchema(
        name="id",
        dtype=DataType.INT64,
        is_primary=True,
        auto_id=True
    ),

    FieldSchema(
        name="text",
        dtype=DataType.VARCHAR,
        max_length=5000
    ),

    FieldSchema(
        name="chunk_type",
        dtype=DataType.VARCHAR,
        max_length=100
    ),

    FieldSchema(
        name="embedding",
        dtype=DataType.FLOAT_VECTOR,
        dim=768
    )
]

schema = CollectionSchema(fields)

collection = Collection(
    name="nutrition_rag",
    schema=schema
)

C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\126301732.py:38: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(


In [187]:
data = []

semantic_embeddings = []
for s_chunks in similarity_chunks:
    embeddings = convert_embeddings(prompt=s_chunks)
    semantic_embeddings.append(embeddings)
    
    data.append([
        s_chunks,
        "semantic",
        embedding
    ])


In [ ]:
def insert_data(collection, chunks=None, chunk_type=None):
    if not chunks:
        print("Error: No chunks provided!")
        return 0, "No data to insert"
    
    texts = []
    chunk_types = []
    embeddings = []

    for i, chunk in enumerate(chunks):
        try:
            converted_embed = convert_embeddings(prompt=chunk)
            texts.append(chunk)
            chunk_types.append(chunk_type)
            embeddings.append(converted_embed)

        except Exception as e:
            print(f"Error processing chunk {i}: {e}")

    print(f"Preparing to insert {len(texts)} records...")
    
    try:
        result = collection.insert([texts, chunk_types, embeddings])
        print(f"Insert result: {result}")
        
        # Flush to ensure data is persisted
        collection.flush()
        print(f"Entities after flush: {collection.num_entities}")
        
        return collection.num_entities, "Successfully inserted data into Milvus collection"
    
    except Exception as e:
        print(f"Insert failed: {e}")
        return 0, f"Insert failed: {str(e)}"

In [201]:
# Check collection info before insert
print(f"Collection name: {collection.name}")
print(f"Collection schema: {collection.schema}")
print(f"Entities before insert: {collection.num_entities}")
print(f"Number of chunks to insert: {len(similarity_chunks)}")
print(f"First chunk preview: {similarity_chunks[0][:100]}...")

# Insert data
num_entities, message = insert_data(collection, chunks=similarity_chunks, chunk_type="semantic")
print(f"\nResult: {num_entities} entities - {message}")

Collection name: nutrition_rag
Collection schema: {'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 5000}}, {'name': 'chunk_type', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False, 'enable_namespace': False}
Entities before insert: 276
Number of chunks to insert: 138
First chunk preview: 1
FACT SHEET N°394 
UPDATED AUGUST 2018
Healthy diet
KEY FACTS
n A healthy diet helps to protect aga...
First embedding shape: (768,)


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\3592652342.py:4: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"Entities before insert: {collection.num_entities}")


Preparing to insert 138 records...


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:25: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  result = collection.insert([texts, chunk_types, embeddings])


Insert result: (insert count: 138, delete count: 0, upsert count: 0, timestamp: 466353675826102289, success count: 138, err count: 0


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:29: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.flush()


Entities after flush: 414

Result: 414 entities - Successfully inserted data into Milvus collection


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:30: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"Entities after flush: {collection.num_entities}")
C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:32: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  return collection.num_entities, "Successfully inserted data into Milvus collection"


In [202]:
# Check collection info before insert
print(f"Collection name: {collection.name}")
print(f"Collection schema: {collection.schema}")
print(f"Entities before insert: {collection.num_entities}")
print(f"Number of chunks to insert: {len(similarity_chunks)}")
print(f"First chunk preview: {similarity_chunks[0][:100]}...")

# Insert data
num_entities, message = insert_data(collection, chunks=chunks, chunk_type="recursive_character")
print(f"\nResult: {num_entities} entities - {message}")

Collection name: nutrition_rag
Collection schema: {'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 5000}}, {'name': 'chunk_type', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False, 'enable_namespace': False}
Entities before insert: 414
Number of chunks to insert: 138
First chunk preview: 1
FACT SHEET N°394 
UPDATED AUGUST 2018
Healthy diet
KEY FACTS
n A healthy diet helps to protect aga...
First embedding shape: (768,)


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\313022692.py:4: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"Entities before insert: {collection.num_entities}")


Preparing to insert 23 records...


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:25: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  result = collection.insert([texts, chunk_types, embeddings])


Insert result: (insert count: 23, delete count: 0, upsert count: 0, timestamp: 466353746127355921, success count: 23, err count: 0


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:29: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.flush()


Entities after flush: 437

Result: 437 entities - Successfully inserted data into Milvus collection


C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:30: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(f"Entities after flush: {collection.num_entities}")
C:\Users\Aathi K M\AppData\Local\Temp\ipykernel_33268\1126694712.py:32: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  return collection.num_entities, "Successfully inserted data into Milvus collection"
